# Feature Preparation

Builds the 10-feature vector for RUL modeling and saves it to `Features/`. Plots of the result are in `main.ipynb`.

Every fitted step (regime baseline, Fisher selection, PCA) uses the training engines only, then runs unchanged on the validation and test engines. `GroundTruth/` is not used here.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.io as sio
from IPython.display import display
from sklearn.decomposition import PCA

HEALTHY_FLIGHTS = 30   # first flights of each engine, assumed healthy
LATE_FLIGHTS = 30      # last flights of each engine, treated as degraded
SMOOTH_FLIGHTS = 15    # trailing window for smoothed sensors and health
WINDOW_FLIGHTS = 20    # start-of-life and slope window; must fit the shortest test engine (21 flights)
VAL_ENGINES = 52       # 20% of the 260 training engines
SPEEDS = ["Nf", "NRf", "Nc", "NRc"]   # trend inconsistently across engines (main.ipynb, plot 5)

## Load the training and test engines

In [2]:
df = pd.read_csv("TrainingData_Engine.csv", encoding="utf-8")
df["RUL"] = df.groupby("EngineID")["Cycle"].transform("max") - df["Cycle"]

names = {c: c.split()[0] for c in df.columns[6:] if c != "RUL"}
sensors = list(names.values())
data = df.rename(columns=names)

mat = sio.loadmat("Testing_DataSet/TestingData_Engine.mat", squeeze_me=True, struct_as_record=False)
assert [str(v).split()[0] for v in mat["Variable_List"]] == [c.split()[0] for c in df.columns[3:27]]

test_runs = []
for engine_id, run in enumerate(mat["Testing_EngineRun"], start=1):
    frame = pd.DataFrame(run.Data, columns=df.columns[3:27]).rename(columns=names)
    frame.insert(0, "EngineID", engine_id)
    frame.insert(1, "Cycle", np.arange(1, len(frame) + 1))
    frame.insert(2, "Regime", run.Regime.astype(int))
    test_runs.append(frame)
test_data = pd.concat(test_runs, ignore_index=True)

print(f"Training: {data['EngineID'].nunique()} engines, {len(data):,} flights")
print(f"Test: {test_data['EngineID'].nunique()} engines, {len(test_data):,} flights")

Training: 260 engines, 53,759 flights
Test: 100 engines, 12,817 flights


## Split by engine

Whole engines are held out, so no engine has flights on both sides of the split. The test files reuse IDs 1-100, so an ID only identifies an engine within its own split.

In [3]:
engine_ids = data["EngineID"].unique()
val_ids = np.random.default_rng(0).choice(engine_ids, VAL_ENGINES, replace=False)

val_data = data[data["EngineID"].isin(val_ids)].reset_index(drop=True)
train_data = data[~data["EngineID"].isin(val_ids)].reset_index(drop=True)

assert not set(train_data["EngineID"]) & set(val_data["EngineID"])
lives = {name: frame.groupby("EngineID")["Cycle"].max() for name, frame in [("train", train_data), ("val", val_data)]}
display(pd.DataFrame(lives).describe().loc[["count", "min", "50%", "mean", "max"]].round(1))

,train,val
count,208.0,52.0
min,128.0,145.0
50%,198.5,200.5
mean,205.8,210.5
max,365.0,378.0


## Fit on the training engines only

In [4]:
baseline = train_data[train_data["Cycle"] <= HEALTHY_FLIGHTS].groupby("Regime")[sensors].agg(["mean", "std"])

def normalize(frame):
    """Z-score each sensor within its regime; sensors flat within a regime become 0."""
    mean = baseline.xs("mean", axis=1, level=1).reindex(frame["Regime"]).to_numpy()
    std = baseline.xs("std", axis=1, level=1).reindex(frame["Regime"]).to_numpy()
    z = (frame[sensors].to_numpy() - mean) / np.where(std > 1e-9, std, np.inf)
    return pd.DataFrame(z, columns=sensors, index=frame.index)

z_train = normalize(train_data)

healthy = z_train[train_data["Cycle"] <= HEALTHY_FLIGHTS]
late = z_train[train_data["RUL"] < LATE_FLIGHTS]
fisher = ((late.mean() - healthy.mean()) ** 2 / (late.var() + healthy.var())).fillna(0).sort_values(ascending=False)
selected = [s for s in fisher.index[fisher >= 2] if s not in SPEEDS]

pca = PCA().fit(z_train[selected])   # all components kept for reference; only PC1 is used
pc1_train = pca.transform(z_train[selected])[:, 0]
PC1_SIGN = -np.sign(np.corrcoef(pc1_train, train_data["RUL"])[0, 1])   # higher PC1 means more degraded

print(f"Selected sensors: {selected}")
print(f"PC1 explains {pca.explained_variance_ratio_[0]:.1%} of their variance")
display(fisher.head(8).round(2).to_frame("Fisher score"))

Selected sensors: ['Ps30', 'T50', 'BPR', 'htBleed', 'T24', 'T30']
PC1 explains 71.0% of their variance


,Fisher score
Ps30,6.26
T50,5.10
BPR,4.51
htBleed,3.16
T24,2.82
T30,2.70
phi,1.45
P30,1.28


## Principal components

All six components of the selected sensors, fitted on the training engines. Each row lists a component's share of variance and its loading on each sensor.

Only PC1 is used in the features, via `health`, `health_delta` and `health_slope`. Its sign is flipped so that higher means more degraded. The signs of PC2-PC6 are arbitrary.

In [5]:
loadings = pca.components_.copy()
loadings[0] *= PC1_SIGN
components = pd.DataFrame(loadings, columns=selected, index=[f"PC{i}" for i in range(1, len(selected) + 1)])
components.insert(0, "variance explained", pca.explained_variance_ratio_)
components.insert(1, "cumulative", pca.explained_variance_ratio_.cumsum())
display(components.style.format("{:.3f}").format("{:.1%}", subset=["variance explained", "cumulative"]))

,variance explained,cumulative,Ps30,T50,BPR,htBleed,T24,T30
PC1,71.0%,71.0%,0.494,0.443,0.444,0.361,0.348,0.334
PC2,6.8%,77.8%,-0.043,-0.050,-0.088,-0.454,-0.130,0.874
PC3,6.6%,84.5%,-0.061,-0.049,-0.184,-0.397,0.890,-0.098
PC4,6.5%,90.9%,-0.204,-0.186,-0.549,0.693,0.206,0.315
PC5,5.0%,95.9%,0.142,0.753,-0.601,-0.140,-0.147,-0.105
PC6,4.1%,100.0%,0.830,-0.444,-0.315,-0.073,-0.072,-0.065


## Build the features

| Feature | Definition |
|---|---|
| 6 sensors | Regime-normalized, trailing 15-flight mean |
| `health` | Trailing 15-flight mean of PC1 |
| `health_delta` | `health` minus the engine's mean PC1 over its first 20 flights |
| `health_slope` | Linear slope of PC1 over the last 20 flights |
| `Cycle` | Flights so far |

`health_slope` is fitted to unsmoothed PC1. Fitting it to `health` would need 34 flights (15 to smooth, then 20 to fit, sharing one), and the shortest test engine has 21.

Rows start at flight 20, the first flight where all 10 features use past data only. `RUL` is left uncapped so that a cap can be tried at modeling time.

In [6]:
slope_x = np.arange(WINDOW_FLIGHTS) - (WINDOW_FLIGHTS - 1) / 2
slope_weights = slope_x / (slope_x ** 2).sum()

def build_features(frame):
    """Return one row per flight from WINDOW_FLIGHTS onward, using only that flight and earlier ones."""
    z = normalize(frame)
    engine = frame["EngineID"]
    pc1 = pd.Series(PC1_SIGN * pca.transform(z[selected])[:, 0], index=frame.index)

    out = frame[["EngineID", "Cycle"]].copy()
    for sensor in selected:
        out[sensor] = z[sensor].groupby(engine).transform(lambda s: s.rolling(SMOOTH_FLIGHTS).mean())
    out["health"] = pc1.groupby(engine).transform(lambda s: s.rolling(SMOOTH_FLIGHTS).mean())
    start = pc1.where(frame["Cycle"] <= WINDOW_FLIGHTS).groupby(engine).transform("mean")
    out["health_delta"] = out["health"] - start
    out["health_slope"] = pc1.groupby(engine).transform(
        lambda s: s.rolling(WINDOW_FLIGHTS).apply(lambda y: slope_weights @ y, raw=True))
    if "RUL" in frame:
        out["RUL"] = frame["RUL"]
    return out[out["Cycle"] >= WINDOW_FLIGHTS].reset_index(drop=True)

features = {"train": build_features(train_data), "val": build_features(val_data), "test": build_features(test_data)}
FEATURES = selected + ["health", "health_delta", "health_slope", "Cycle"]

X_train, y_train = features["train"][FEATURES], features["train"]["RUL"]
X_val, y_val = features["val"][FEATURES], features["val"]["RUL"]
test_last = features["test"].groupby("EngineID").tail(1).reset_index(drop=True)   # the flight each prediction is made from
X_test = test_last[FEATURES]

print(f"X_train {X_train.shape}, X_val {X_val.shape}, X_test {X_test.shape}")
features["train"].head()

X_train (38860, 10), X_val (9959, 10), X_test (100, 10)


,EngineID,Cycle,Ps30,T50,BPR,htBleed,T24,T30,health,health_delta,health_slope,RUL
0,5,20,-0.290274,-0.480467,-0.631456,-0.521860,-0.179283,-0.262733,-3.176455,0.114583,0.047016,134
1,5,21,-0.344855,-0.540578,-0.514745,-0.403548,-0.216955,-0.101928,-3.094989,0.196050,0.045242,133
2,5,22,-0.335743,-0.503581,-0.468320,-0.356083,-0.300525,-0.065924,-3.053380,0.237659,0.041841,132
3,5,23,-0.331211,-0.495937,-0.565800,-0.342680,-0.241553,-0.139151,-3.090132,0.200906,0.026399,131
4,5,24,-0.234884,-0.508461,-0.589296,-0.422476,-0.227669,-0.079029,-3.062410,0.228628,0.021994,130


## Check the feature vector

Truncating an engine and rebuilding its features must reproduce the same row, which shows that no feature looks ahead. Test engines stop mid-life, so a feature that used later flights would behave differently on them.

In [7]:
assert not any(frame[FEATURES].isna().any().any() for frame in features.values())
assert len(X_test) == 100 and test_last["EngineID"].is_unique

for engine_id, flight in [(train_data["EngineID"].iloc[0], 60), (val_ids[0], 25), (val_ids[1], WINDOW_FLIGHTS)]:
    source = data[data["EngineID"] == engine_id]
    truncated = build_features(source[source["Cycle"] <= flight].reset_index(drop=True)).iloc[-1]
    full = build_features(source.reset_index(drop=True)).query("Cycle == @flight").iloc[0]
    assert np.allclose(truncated[FEATURES].astype(float), full[FEATURES].astype(float))

low, high = X_train.min(), X_train.max()
summary = pd.DataFrame({
    "train min": low, "train max": high,
    "test min": X_test.min(), "test max": X_test.max(),
    "test engines outside train range": ((X_test < low) | (X_test > high)).sum(),
    "Spearman with RUL (train)": X_train.corrwith(y_train, method="spearman"),
    "Spearman with RUL (val)": X_val.corrwith(y_val, method="spearman"),
})
print("All checks passed")
display(summary.round(3))

All checks passed


,train min,train max,test min,test max,test engines outside train range,Spearman with RUL (train),Spearman with RUL (val)
Ps30,-1.700,4.970,-0.994,4.414,0,-0.773,-0.771
T50,-1.572,4.295,-0.894,3.636,0,-0.782,-0.763
BPR,-1.479,4.635,-0.986,3.816,0,-0.775,-0.760
htBleed,-1.487,3.779,-0.827,3.033,0,-0.782,-0.763
T24,-1.402,3.593,-0.625,3.027,0,-0.779,-0.769
T30,-1.480,3.547,-1.115,2.856,0,-0.778,-0.755
health,-5.030,7.230,-4.393,5.995,0,-0.793,-0.779
health_delta,-0.681,11.049,-0.041,9.265,0,-0.904,-0.876
health_slope,-0.120,0.319,-0.088,0.215,0,-0.674,-0.667
Cycle,20.000,365.000,21.000,367.000,1,-0.765,-0.687


## Save

`features_test.csv` keeps every test flight for trajectory methods, and `features_test_last.csv` holds the one row per engine that a prediction is made from.

In [8]:
out_dir = Path("Features")
out_dir.mkdir(exist_ok=True)
for split, frame in features.items():
    frame.to_csv(out_dir / f"features_{split}.csv", index=False)
test_last.to_csv(out_dir / "features_test_last.csv", index=False)

print(*sorted(p.name for p in out_dir.iterdir()), sep="\n")

features_test.csv
features_test_last.csv
features_train.csv
features_val.csv
